In [3]:
import csv
import pandas as pd
import pyperclip
import numpy as np
import math
import warnings
import re

warnings.filterwarnings("ignore")

In [5]:
date_str = '2025-09-24'

In [7]:
#read in the list of courses...Louis helped to scrape this data from the web
#https://bannerssb.utk.edu/kbanpr/bwckgens.p_proc_term_date
df = pd.read_excel('course_list_fall2025.xlsx')

df.columns = ['Course']
df

,Course
0,Foundations of Accounting - 43202 - ACCT 200 -...
1,Foundations of Accounting - 43203 - ACCT 200 -...
2,Foundations of Accounting - 43204 - ACCT 200 -...
3,Foundations of Accounting - 43205 - ACCT 200 -...
4,Foundations of Accounting - 43206 - ACCT 200 -...
...,...
11091,Special Topics in Global Cinema - 57703 - WLC ...
11092,Proficiency Exam - 57587 - WLC 495 - 001
11093,Proficiency Exam - 57588 - WLC 495 - 002
11094,Teaching a Foreign Language - 57809 - WLC 512 ...


In [8]:
#parse the data

# Create a pattern using regular expression to split the 'course' column
pattern = r'^(.*?)\s-\s(\d{5})\s-\s(.*?)\s-\s(\d+)$'

# Apply the pattern using str.extract
split_course = df['Course'].str.extract(pattern)

# Rename the columns as needed
split_course.columns = ['Course Title', 'Course Code', 'Additional Info', 'Additional Code']

# Display the updated DataFrame
split_course

,Course Title,Course Code,Additional Info,Additional Code
0,Foundations of Accounting,43202,ACCT 200,001
1,Foundations of Accounting,43203,ACCT 200,002
2,Foundations of Accounting,43204,ACCT 200,003
3,Foundations of Accounting,43205,ACCT 200,004
4,Foundations of Accounting,43206,ACCT 200,005
...,...,...,...,...
11091,Special Topics in Global Cinema,57703,WLC 482,001
11092,Proficiency Exam,57587,WLC 495,001
11093,Proficiency Exam,57588,WLC 495,002
11094,Teaching a Foreign Language,57809,WLC 512,001


In [9]:
# Create new columns by splitting 'Additional Info'
split_info = split_course['Additional Info'].str.split(expand=True)

# Rename the new columns as needed
split_info.columns = ['Info1', 'Info2']

# Concatenate the new columns with the existing DataFrame
split_course = pd.concat([split_course, split_info], axis=1)

# Drop the original 'Additional Info' column
split_course = split_course.drop('Additional Info', axis=1)

split_course = split_course.rename(columns = {'Course Code': 'Unknown_Code', 'Additional Code': 'Section',
                                              'Info1': 'Department', 'Info2': 'Course Number'})


In [10]:
#read in the most recent overlap report (the result of the other part of the weekly bookstore overlap process)

book_list = pd.read_excel(date_str+'_overlap_report.xlsx')

In [11]:
#parse the course codes to match what we did with the scraped course list

# Split the 'Course Code' column into multiple columns
course_split = book_list['Course Code'].str.split('[.-]', expand=True)

# Rename the columns as needed
course_split.columns = ['Term', 'Department', 'Course Number', 'Section', 'CRN']

# Concatenate the new columns with the existing DataFrame
book_list = pd.concat([book_list, course_split], axis=1)

# Drop the original 'Course Code' column
book_list = book_list.drop('Course Code', axis=1)
book_list['readable_course'] = book_list['Department'] + ' - ' + book_list['Course Number'] + ' - ' + \
    book_list['Section']

In [12]:
#merge the course list with the book list

merged = pd.merge(book_list, split_course, how = 'left', left_on = 'CRN', right_on = 'Unknown_Code')

In [13]:
#pd.set_option('display.max_rows', None)
merged['readable_course']

0      ENVE - 512 - 001
1      GEOG - 131 - 001
2      GEOG - 131 - 002
3      GEOG - 131 - 003
4      GEOG - 131 - 004
             ...       
180    ANTH - 480 - 001
181    ANTH - 480 - 002
182    ANTH - 480 - 003
183    ANTH - 480 - 004
184     LAW - 965 - 001
Name: readable_course, Length: 185, dtype: object

In [14]:
merged = merged[['Electronic location', 'Term', 'Department_x', 'Course Number_x', 'Section_x', 'CRN', 'readable_course',
                 'Course Title', 'Short Title', 'MMS ID']]
merged.columns = merged.columns.str.replace('_x', '') 

merged = merged.rename(columns = {'Electronic location': 'Permalink'})

In [15]:
merged = merged.sort_values(by=['Department', 'Course Number', 'Section'])
merged['Short Title'] = merged['Short Title'].str.lower()
merged['Short Title'] = merged['Short Title'].str.title()

In [19]:
# Check MMS IDs for Excel misformatting
merged['MMS ID'] = merged['MMS ID'].astype(str)

for i in range(len(merged['MMS ID'])):
    merged['MMS ID'].iloc[i] = merged['MMS ID'].iloc[i][0:15] + '1'

In [22]:
# Fill in the Primo Link based on the MMS ID

for i in range(len(merged['MMS ID'])):
    merged['Permalink'].iloc[i] = 'https://utk.primo.exlibrisgroup.com/permalink/01UTN_KNOXVILLE/bcmt7h/alma' + str(merged['MMS ID'].iloc[i])

In [27]:
# Clean up Title Names
merged['Clean Title'] = merged['Short Title'].str.split(':', expand = True)[0]

for i in range(len(merged['Clean Title'])):
    merged['Clean Title'].iloc[i] = re.sub("[\(\[].*?[\)\]]", "", merged['Clean Title'].iloc[i]).strip()

merged

,Permalink,Term,Department,Course Number,Section,CRN,readable_course,Course Title,Short Title,MMS ID,Clean Title
174,https://utk.primo.exlibrisgroup.com/permalink/...,202540,ADVT,490,002,45062,ADVT - 490 - 002,Special Topics,The Art Of The Pitch : Persuasion And Presenta...,9926737922302311,The Art Of The Pitch
94,https://utk.primo.exlibrisgroup.com/permalink/...,202540,AFST,456,001,47492,AFST - 456 - 001,"Race, Ethnicity, Crime, and Justice",The New Jim Crow : Mass Incarceration In The A...,9925921467802311,The New Jim Crow
95,https://utk.primo.exlibrisgroup.com/permalink/...,202540,AFST,456,002,58261,AFST - 456 - 002,"Race, Ethnicity, Crime, and Justice",The New Jim Crow : Mass Incarceration In The A...,9925921467802311,The New Jim Crow
180,https://utk.primo.exlibrisgroup.com/permalink/...,202540,ANTH,480,001,40311,ANTH - 480 - 001,Human Osteology,Human Osteology,9926677862202311,Human Osteology
181,https://utk.primo.exlibrisgroup.com/permalink/...,202540,ANTH,480,002,40312,ANTH - 480 - 002,Human Osteology,Human Osteology,9926677862202311,Human Osteology
...,...,...,...,...,...,...,...,...,...,...,...
140,https://utk.primo.exlibrisgroup.com/permalink/...,202540,TPTE,617,001,49257,TPTE - 617 - 001,Advanced Studies in Education - An Interdiscip...,The Amateur Hour : A History Of College Teachi...,9926307357302311,The Amateur Hour
56,https://utk.primo.exlibrisgroup.com/permalink/...,202540,WGS,320,001,50044,WGS - 320 - 001,"Gender, Sexuality, and Religion",Queer Religiosities : An Introduction To Queer...,9926026155802311,Queer Religiosities
57,https://utk.primo.exlibrisgroup.com/permalink/...,202540,WGS,320,002,55841,WGS - 320 - 002,"Gender, Sexuality, and Religion",Queer Religiosities : An Introduction To Queer...,9926026155802311,Queer Religiosities
100,https://utk.primo.exlibrisgroup.com/permalink/...,202540,WGS,332,001,49943,WGS - 332 - 001,Women in American Literature,Ceremony,9925919468802311,Ceremony


In [30]:
#pd.set_option('display.max_rows', None)

In [31]:
# Auto-fix title capitalization issues before exporting

for i in range(merged.shape[0]):
    if merged.at[i, 'Short Title'] == 'Genki Ii Workbook':
        merged.at[i, 'Clean Title'] = 'Genki II Workbook'
    elif merged.at[i, 'Short Title'] == 'Genki Ii':
        merged.at[i, 'Clean Title'] = 'Genki II'
    elif merged.at[i, 'Short Title'] ==  "Jacob'S Room":
        merged.at[i, 'Clean Title'] = "Jacob's Room"
    elif merged.at[i, 'Short Title'] ==  "A Writer'S Diary : The Virginia Woolf Library Authorized Edition":
        merged.at[i, 'Clean Title'] = "A Writer's Diary : The Virginia Woolf Library Authorized Edition"
    elif merged.at[i, 'Short Title'] ==  "Madam C. J. Walker'S Gospel Of Giving : Black Women'S Philanthropy During Jim Crow":
        merged.at[i, 'Clean Title'] = "Madam C. J. Walker's Gospel Of Giving"
    elif merged.at[i, 'Short Title'] == "Unlocking Multilingual Learners? Potential":
        merged.at[i, 'Clean Title'] = "Unlocking Multilingual Learners' Potential"
    elif merged.at[i, 'Short Title'] == "Wetzel'S Limnology : Lake And River Ecosystems":
        merged.at[i, 'Clean Title'] = "Wetzel's Limnology"
    elif merged.at[i, 'Course Title'] == "Linguistics of American Sign Language":
        merged.at[i, 'Clean Title'] = "Linguistics of ASL"
    elif merged.at[i, 'Short Title'] == "Where'S The Math? : Books, Games, And Routines To Spark Children'S Thinking":
        merged.at[i, 'Clean Title'] = "Where's the Math?"
    elif merged.at[i, 'Short Title'] == "Why Kids Can'T Read : Continuing To Challenge The Status Quo In Education":
        merged.at[i, 'Clean Title'] = "Why Kids Can't Read"
    elif merged.at[i, 'Short Title'] == 'Making And Tinkering With Stem : Solving Design Challenges With Young Children':
        merged.at[i, 'Clean Title'] = 'Making and Tinkering with STEM'





# Auto-fill missing Course Titles before exporting

for i in range(merged.shape[0]):
    if merged.at[i, 'readable_course'] == 'AFST - 336 - 001' or merged.at[i, 'readable_course'] == 'ENGL - 336 - 001':
        merged.at[i, 'Course Title'] = 'Caribbean Literature'
    elif merged.at[i, 'readable_course'] == 'AFST - 329 - 001':
        merged.at[i, 'Course Title'] = 'Black History in Tennessee 1796-1980'
    elif merged.at[i, 'readable_course'] == 'MATH - 431 - 001':
        merged.at[i, 'Course Title'] = 'Differential Equations II'
    elif merged.at[i, 'readable_course'] == 'MATH - 435 - 001':
        merged.at[i, 'Course Title'] = 'Partial Differential Equations'
    elif merged.at[i, 'readable_course'] == 'ASL - 421 - 502' or merged.at[i, 'readable_course'] == 'ASL - 421 - 301':
        merged.at[i, 'Course Title'] = 'History & Culture of the Deaf'
    elif merged.at[i, 'readable_course'] == 'EDDE - 415 - 001':
        merged.at[i, 'Course Title'] = 'Language Development of Deaf/Hard of Hearing I'
    elif merged.at[i, 'readable_course'] == 'EEPS - 104 - 012':
        merged.at[i, 'Course Title'] = 'Exploring the Planets'
    elif merged.at[i, 'readable_course'] == 'ENGL - 298 - 013':
        merged.at[i, 'Course Title'] = 'Honors Writing and Research'
    elif merged.at[i, 'readable_course'] == 'ENGL - 446 - 001':
        merged.at[i, 'Course Title'] = 'Native American Literature'
    elif merged.at[i, 'Department'] == 'GEOG' and merged.at[i, 'Course Number'] == '131':
        merged.at[i, 'Course Title'] = 'Weather, Climate, and Climate Change'
    elif merged.at[i, 'readable_course'] == 'REED - 529 - 003':
        merged.at[i, 'Course Title'] = 'Assessment and Instruction with Emergent Learners PreK-2'
    elif merged.at[i, 'Department'] == 'REST' and merged.at[i, 'Course Number'] == '375':
        merged.at[i, 'Course Title'] = 'Theravada Buddhism'
    elif merged.at[i, 'readable_course'] == 'SOWK - 562 - 018':
        merged.at[i, 'Course Title'] = 'Interpersonal Practice with Adult Individuals'


merged

,Permalink,Term,Department,Course Number,Section,CRN,readable_course,Course Title,Short Title,MMS ID,Clean Title
174,https://utk.primo.exlibrisgroup.com/permalink/...,202540,ADVT,490,002,45062,ADVT - 490 - 002,Special Topics,The Art Of The Pitch : Persuasion And Presenta...,9926737922302311,The Art Of The Pitch
94,https://utk.primo.exlibrisgroup.com/permalink/...,202540,AFST,456,001,47492,AFST - 456 - 001,"Race, Ethnicity, Crime, and Justice",The New Jim Crow : Mass Incarceration In The A...,9925921467802311,The New Jim Crow
95,https://utk.primo.exlibrisgroup.com/permalink/...,202540,AFST,456,002,58261,AFST - 456 - 002,"Race, Ethnicity, Crime, and Justice",The New Jim Crow : Mass Incarceration In The A...,9925921467802311,The New Jim Crow
180,https://utk.primo.exlibrisgroup.com/permalink/...,202540,ANTH,480,001,40311,ANTH - 480 - 001,Human Osteology,Human Osteology,9926677862202311,Human Osteology
181,https://utk.primo.exlibrisgroup.com/permalink/...,202540,ANTH,480,002,40312,ANTH - 480 - 002,Human Osteology,Human Osteology,9926677862202311,Human Osteology
...,...,...,...,...,...,...,...,...,...,...,...
140,https://utk.primo.exlibrisgroup.com/permalink/...,202540,TPTE,617,001,49257,TPTE - 617 - 001,Advanced Studies in Education - An Interdiscip...,The Amateur Hour : A History Of College Teachi...,9926307357302311,The Amateur Hour
56,https://utk.primo.exlibrisgroup.com/permalink/...,202540,WGS,320,001,50044,WGS - 320 - 001,"Gender, Sexuality, and Religion",Queer Religiosities : An Introduction To Queer...,9926026155802311,Queer Religiosities
57,https://utk.primo.exlibrisgroup.com/permalink/...,202540,WGS,320,002,55841,WGS - 320 - 002,"Gender, Sexuality, and Religion",Queer Religiosities : An Introduction To Queer...,9926026155802311,Queer Religiosities
100,https://utk.primo.exlibrisgroup.com/permalink/...,202540,WGS,332,001,49943,WGS - 332 - 001,Women in American Literature,Ceremony,9925919468802311,Ceremony


In [35]:
merged.at[141, 'Course Title'] # Check Wetzel's Limnology on 2-19

'Exploring the Planets'

In [37]:
merged.shape

(185, 11)

In [39]:
#use the merged dataframe to generate javascript that we will copy/paste into the HTML of the Automagic libguide
#in the existing HTML of the libguide, replace all of the const courseData with what you see in the printed output below
#the line "pyperclip.copy(complete_javascript_array)" automatically puts the necessary javascript onto your clipboard
#all you need to do is paste
#just make sure you replace everything in the html that starts with "const CourseData" and ends with "];"

# Create a list to store the JavaScript objects
javascript_objects = []

# Build the JavaScript array
for index, row in merged.iterrows():
    javascript_objects.append(f'  {{ course: "{row["readable_course"]} - {row["Course Title"]} - {row["Clean Title"]}", permalink: "{row["Permalink"]}" }}')

# Concatenate the JavaScript objects into a single string
javascript_array = ',\n'.join(javascript_objects)

# Form the complete JavaScript array with declaration and closing brackets
complete_javascript_array = f'const courseData = [\n{javascript_array}\n];'

# Print the complete JavaScript array
print(complete_javascript_array)

# Copy the complete JavaScript array to the clipboard
pyperclip.copy(complete_javascript_array)
print("Complete JavaScript array has been copied to the clipboard.")



const courseData = [
  { course: "ADVT - 490 - 002 - Special Topics - The Art Of The Pitch", permalink: "https://utk.primo.exlibrisgroup.com/permalink/01UTN_KNOXVILLE/bcmt7h/alma9926737922302311" },
  { course: "AFST - 456 - 001 - Race, Ethnicity, Crime, and Justice - The New Jim Crow", permalink: "https://utk.primo.exlibrisgroup.com/permalink/01UTN_KNOXVILLE/bcmt7h/alma9925921467802311" },
  { course: "AFST - 456 - 002 - Race, Ethnicity, Crime, and Justice - The New Jim Crow", permalink: "https://utk.primo.exlibrisgroup.com/permalink/01UTN_KNOXVILLE/bcmt7h/alma9925921467802311" },
  { course: "ANTH - 480 - 001 - Human Osteology - Human Osteology", permalink: "https://utk.primo.exlibrisgroup.com/permalink/01UTN_KNOXVILLE/bcmt7h/alma9926677862202311" },
  { course: "ANTH - 480 - 002 - Human Osteology - Human Osteology", permalink: "https://utk.primo.exlibrisgroup.com/permalink/01UTN_KNOXVILLE/bcmt7h/alma9926677862202311" },
  { course: "ANTH - 480 - 003 - Human Osteology - Human Osteolo